1. Importing Necessary Libraries

In [31]:
import os, re
from pathlib import Path
import numpy as np
import numpy as _np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
print("TF:", tf.__version__)

TF: 2.20.0


2. Configuration 

In [32]:
# Define model paths, dataset annotation settings, preprocessing parameters and output locations for the evaluation pipeline.

CONFIG = {
    # Models to evaluate (name -> file path) 
    "models": {
        "ANN": "models/ann_detector.keras",
        "CNN": "models/cnn_detector.keras",
        "RNN": "models/rnn_detector.keras",
        "SEMI_CNN": "models/model_v_semi.keras",
        "SEMI_CNN_LIFELONG": "models/teacher_ep3.keras",
    },
    # Annotation / dataset description 
    "annotation": {
        "type": "csv",  # Input metadata type (currently expecting a CSV file)
        #"path": r"C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/train_detector_annotations.csv",
        #"path": r"C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/unkonwn_adversarial_fgsm_RNN_detector_annotations.csv",
        "path": r"C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection/unkonwn_adversarial_PGD_detector_annotations.csv",
        "base_dir": "",   # Optional root folder that will be prepended to relative image paths
        "filename_template": "{image_id}",  # How to build file paths from CSV rows (e.g. uses 'image_id' column)
        "label_preference": ["adv_label", "label"],     # Columns to try (in order) for the binary label
        "test_value": "test",   # Value in the split column that designates TEST rows
        "aux_columns": ["isNight"],   # Extra numeric features to append (e.g., 0/1 for night)
    },
    # Image preprocessing 
    "img_size": 50,         # Images will be resized to (img_size, img_size)
    "num_channels": 3,      # 3 for RGB, 1 for grayscale
    "batch_size": 64,       # Used if you batch-load / predict in chunks

    # Output directory (Excel and any optional artifacts) 
    "out_dir": ".",
}
print("CONFIG loaded.")

CONFIG loaded.


3. Utility & Data Loading Helpers

In [33]:
# Provide filesystem helpers, CSV parsing, Unicode-safe image loading, input tensor assembly for different model input shapes, and a minimal
# Excel aggregator for binary metrics. This section does NOT write any PNG/CSV artifacts; only the Excel writer is used elsewhere.        

def ensure_out_dir():
    """Create (if missing) and return the output directory Path from CONFIG['out_dir']."""
    p = Path(CONFIG.get("out_dir","eval_outputs"))
    p.mkdir(parents=True, exist_ok=True)
    return p

def _infer_col(df, choices):
    """Return the first existing column from `choices` (case-insensitive)."""
    for c in choices:
        if c in df.columns: return c
    lowers = {c.lower(): c for c in df.columns}
    for c in choices:
        if c.lower() in lowers: return lowers[c.lower()]
    return None

def _infer_split_col(df):
    """Best-effort detection of a 'split' column (e.g., split/subset/partition/set)."""
    for c in ["split","subset","partition","set","split_type","phase","dataset","usage","fold"]:
        if c in df.columns: return c
    for c in df.columns:
        cl = c.lower()
        if "split" in cl or "subset" in cl or "partition" in cl:
            return c
    return None

def _to_int_label(v):
    """
    Convert a label to {0,1}. Supports common textual forms:
    - Positive: '1', 'true', 'yes', 'attack', 'adv', 'adversarial', 'malicious'
    - Negative: '0', 'false', 'no', 'clean', 'benign', 'normal'
    Falls back to int(float(v)) if string is numeric-like; returns NaN on failure.
    """
    try:
        if isinstance(v, str):
            s = v.strip().lower()
            if s in ("1","true","yes","attack","adv","adversarial","malicious"): return 1
            if s in ("0","false","no","clean","benign","normal"): return 0
            return int(float(s))
        return int(v)
    except Exception:
        return np.nan

def _to_float_aux(v):
    """
    Convert auxiliary feature values to float.
    Accepts ints/floats/bools/strings like 'true'/'false'/'1'/'0', else returns 0.0.
    NOTE: This implementation references `_np` aliases. Ensure you either:
      - `import numpy as _np` earlier, OR
      - replace `_np` usages with `np` if you prefer.
    """
    if isinstance(v, (int,float,_np.integer,_np.floating)): return float(v)
    if isinstance(v, (bool,_np.bool_)): return 1.0 if v else 0.0
    if isinstance(v,str):
        s=v.strip().lower()
        if s in ("1","true","yes","y"): return 1.0
        if s in ("0","false","no","n"): return 0.0
        try: return float(s)
        except: return 0.0
    try: return float(v)
    except: return 0.0

def _build_path_from_row(row, base_dir, tmpl):
    """
    Build an absolute/normalized file path for an image row.
    Priority:
      1) Direct path columns: ['path','filepath','file','image_path','image','filename']
      2) Use `filename_template` with row fields (e.g., '{image_id}')
      3) Fallback to 'image_id' if present
    """
    for c in ["path","filepath","file","image_path","image","filename"]:
        v = row.get(c, None)
        if isinstance(v,str) and v.strip(): return os.path.normpath(v)
    if tmpl and "{" in tmpl and "}" in tmpl:
        try:
            rel = tmpl.format(**row)
            return os.path.normpath(os.path.join(base_dir, rel) if base_dir else rel)
        except: pass
    if "image_id" in row and isinstance(row["image_id"], str):
        rel = row["image_id"]
        return os.path.normpath(os.path.join(base_dir, rel) if base_dir else rel)
    return None

def load_df_TEST_only(ann_cfg: dict) -> pd.DataFrame:
    """
    Load the annotation CSV, filter to TEST split, normalize label to {0,1},
    and derive a file path column '__file'.
    Raises if a label column cannot be found.
    """
    assert ann_cfg.get("type","csv") == "csv", "Only CSV annotations supported here."
    df = pd.read_csv(ann_cfg["path"])
    # Filter to TEST rows
    split_col = _infer_split_col(df)
    tv = str(ann_cfg.get("test_value","test")).lower()
    if split_col is not None:
        df = df[df[split_col].astype(str).str.lower()==tv].copy()
    else:
        print("⚠️ No split column — using all rows as TEST.")

    # Locate label col
    label_col = None
    for c in ann_cfg.get("label_preference", ["label"]):
        if c in df.columns: label_col=c; break
    if label_col is None:
        label_col = _infer_col(df, ["label","adv_label","y","target"])
    if label_col is None:
        raise ValueError("Label column not found in annotation CSV.")
    
    # Normalize labels
    df["__label"] = df[label_col].apply(_to_int_label).astype("float")
    df = df.dropna(subset=["__label"]).copy()
    df["__label"] = df["__label"].astype(int)

    # Build file paths
    rows = df.to_dict("records")
    base_dir = ann_cfg.get("base_dir","")
    tmpl = ann_cfg.get("filename_template","{image_id}")
    files = [_build_path_from_row(r, base_dir, tmpl) for r in rows]
    df["__file"] = files
    df = df[df["__file"].notna()].copy()
    return df

# Unicode-safe image IO & preprocessing 

def imread_unicode(path: str, num_channels=3):
    """
    Read an image from a Unicode path using cv2.imdecode for robust Windows/Unicode handling.
    Returns HxW[xC] ndarray; raises if decoding fails.
    """
    data = np.fromfile(path, dtype=np.uint8)
    flag = cv2.IMREAD_COLOR if num_channels==3 else cv2.IMREAD_GRAYSCALE
    img = cv2.imdecode(data, flag)
    if img is None: raise FileNotFoundError(f"imdecode failed: {path}")
    return img

def preprocess_img(img, size, num_channels):
    """Resize and normalize image to [0,1], ensuring correct channel count (C=1 or 3)."""
    if num_channels==3 and img.ndim==2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if num_channels==1 and img.ndim==3: img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (size,size), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32)/255.0
    if num_channels==1 and img.ndim==2: img = img[...,None]
    return img

def build_inputs(files, df, size, num_channels, aux_cols):
    """
    Construct three input representations from the TEST set:
      - X_img  : (N, H, W, C) normalized images
      - X_flat : (N, H*W*C) flattened images
      - X_aux  : (N, K) auxiliary numeric features (optional)
    """
    N=len(files); H=W=size; C=num_channels; k=len(aux_cols)
    X_img = np.zeros((N,H,W,C), dtype=np.float32)
    X_flat= np.zeros((N,H*W*C), dtype=np.float32)
    X_aux = np.zeros((N,k), dtype=np.float32) if k>0 else None
    rows = df.to_dict("records")
    for i,(p,r) in enumerate(zip(files, rows)):
        img = preprocess_img(imread_unicode(p, num_channels), size, num_channels)
        X_img[i]=img; X_flat[i]=img.reshape(-1)
        if k: X_aux[i]=[ _to_float_aux(r.get(c,0)) for c in aux_cols ]
    return X_img, X_flat, X_aux

def describe_model_inputs(model):
    """Return a list of input specs (shape/rank) for a Keras model (handles single/multi-input)."""
    ins = model.inputs if isinstance(model.inputs,(list,tuple)) else [model.input]
    specs=[]
    for t in ins:
        sh = tuple(int(d) if d is not None else None for d in t.shape)
        specs.append({"shape":sh, "rank":len(sh)})
    return specs

def _make_seq_from_image(X_img, target_T=None, target_F=None, H=None, W=None, C=None):
    """
    Convert 4D image tensor (N,H,W,C) to 3D sequence (N,T,F).
    Default: T=H, F=W*C (e.g., 50 x (50*3)=150). If T,F given, reshape accordingly.
    """
    N = X_img.shape[0]
    if H is None or W is None or C is None:
        H, W, C = X_img.shape[1:4]
    # default guess: T = H, F = W*C
    T = target_T if target_T is not None else H
    F = target_F if target_F is not None else (W*C)
    # Sanity: if H*W*C mismatches T*F, fall back to flat reshape
    if H*W*C != T*F:
        return X_img.reshape(N, H*W*C)[:, :T*F].reshape(N, T, F)
    return X_img.reshape(N, H, W*C) if (T==H and F==W*C) else X_img.reshape(N, H*W*C).reshape(N, T, F)

def assemble_inputs(model, X_img, X_flat, X_aux):
    """
    Assemble the correct input(s) for a given model based on its expected input shapes.
    Supports:
      - 4D image input (N,H,W,C)
      - 3D sequence input (N,T,F) derived from image
      - 2D inputs (flat, aux, or flat+aux)
      - Two-input models (any combination of the above)
    """
    specs = describe_model_inputs(model)
    H,W,C = X_img.shape[1:4]; d_flat = X_flat.shape[1]; d_aux = 0 if X_aux is None else X_aux.shape[1]
    X_seq_default = X_img.reshape(X_img.shape[0], H, W*C)  # (N, H, W*C) e.g. (N,50,150)

    if len(specs)==1:
        sp=specs[0]
        if sp["rank"]==4:
            return X_img
        if sp["rank"]==3:
            # build sequence with (T,F) from spec if provided
            T = sp["shape"][1]
            F = sp["shape"][2]
            x_seq = _make_seq_from_image(X_img, target_T=T, target_F=F, H=H, W=W, C=C)
            return x_seq
        if sp["rank"]==2:
            dim=sp["shape"][-1]
            if dim is None:
                return np.concatenate([X_flat,X_aux],1) if d_aux>0 else X_flat
            if d_aux>0 and d_flat+d_aux==dim: return np.concatenate([X_flat,X_aux],1)
            if d_flat==dim: return X_flat
            if d_aux>0 and d_aux==dim: return X_aux
            return np.concatenate([X_flat,X_aux],1) if d_aux>0 else X_flat
        raise ValueError(f"Unsupported input rank: {sp}")
    elif len(specs)==2:
        x_list=[None,None]
        for i,sp in enumerate(specs):
            if sp["rank"]==4:
                x_list[i]=X_img
            elif sp["rank"]==3:
                T = sp["shape"][1]
                F = sp["shape"][2]
                x_list[i]= _make_seq_from_image(X_img, target_T=T, target_F=F, H=H, W=W, C=C)
            elif sp["rank"]==2:
                dim=sp["shape"][-1]
                if dim is None:
                    x_list[i]=X_aux if (X_aux is not None) else X_flat
                elif X_aux is not None and dim==d_aux:
                    x_list[i]=X_aux
                elif dim==d_flat:
                    x_list[i]=X_flat
                elif X_aux is not None and dim==(d_flat+d_aux):
                    x_list[i]=np.concatenate([X_flat,X_aux],1)
                else:
                    x_list[i]=X_flat  # fallback
            else:
                raise ValueError(f"Unsupported rank in multi-input: {sp}")
        return x_list
    else:
        raise ValueError(f"Model with {len(specs)} inputs is not supported here.")

# Binary metrics aggregator for Excel export 

class BinaryEvalExcel:
    """
    Collect per-model binary metrics and confusion counts, then write to Excel with 3 sheets:
      - Summary         : model, n_samples, accuracy, precision, recall, f1, roc_auc, pr_auc
      - TP_TN_FP_FN     : per-model TP, TN, FP, FN
      - Confusions      : per-model cell counts for 2x2 confusion matrix
    """
    def __init__(self, class_names=("clean","attack")):
        self.class_names=list(class_names)
        self.sum_rows=[]; self.tpfp_rows=[]; self.conf_rows=[]

    def add(self, model, y_true, y_scores=None, y_pred=None, threshold=0.5):
        """Register metrics for one model given y_true and either y_scores or y_pred."""
        y_true=np.asarray(y_true).astype(int).ravel()
        if y_scores is None and y_pred is None: raise ValueError("Provide y_scores or y_pred")
        if y_scores is not None:
            y_scores=np.asarray(y_scores).ravel()
            if y_pred is None: y_pred=(y_scores>=threshold).astype(int)
        else:
            y_pred=np.asarray(y_pred).astype(int).ravel()
        tn,fp,fn,tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
        self.sum_rows.append({
            "model":model,"n_samples":int(len(y_true)),
            "accuracy":float(accuracy_score(y_true,y_pred)),
            "precision":float(precision_score(y_true,y_pred,pos_label=1,zero_division=0)),
            "recall":float(recall_score(y_true,y_pred,pos_label=1,zero_division=0)),
            "f1":float(f1_score(y_true,y_pred,pos_label=1,zero_division=0)),
            "roc_auc":float(roc_auc_score(y_true,y_scores)) if y_scores is not None else float("nan"),
            "pr_auc":float(average_precision_score(y_true,y_scores)) if y_scores is not None else float("nan"),
        })
        self.tpfp_rows.append({"model":model,"TP":int(tp),"TN":int(tn),"FP":int(fp),"FN":int(fn)})

        # Store confusion counts in long form (one row per cell)
        cm=np.array([[tn,fp],[fn,tp]],dtype=int)  # rows=true(0,1) cols=pred(0,1)
        for i_true in (0,1):
            for j_pred in (0,1):
                self.conf_rows.append({"model":model,"true":self.class_names[i_true],"pred":self.class_names[j_pred],"count":int(cm[i_true,j_pred])})
    
    def save_excel(self, path):
        """Write three sheets (Summary, TP_TN_FP_FN, Confusions) to a single Excel file."""
        df1=pd.DataFrame(self.sum_rows)[["model","n_samples","accuracy","precision","recall","f1","roc_auc","pr_auc"]]
        df2=pd.DataFrame(self.tpfp_rows)[["model","TP","TN","FP","FN"]]
        df3=pd.DataFrame(self.conf_rows)[["model","true","pred","count"]]
        path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
        with pd.ExcelWriter(path) as xw:
            df1.to_excel(xw, sheet_name="Summary", index=False)
            df2.to_excel(xw, sheet_name="TP_TN_FP_FN", index=False)
            df3.to_excel(xw, sheet_name="Confusions", index=False)
        display(df1); print(f"✓ Saved:", path)

4. Run Evaluation & Export (Excel)

In [34]:
# Load TEST split, build input tensors, run all models (ANN/CNN/RNN) SEMI_CNN, compute binary metrics in-memory, and write ONLY `evaluation_summary.xlsx`.

# Prepare output folder and load TEST annotations 
out_dir = ensure_out_dir()
df = load_df_TEST_only(CONFIG["annotation"])
files = df["__file"].tolist()
y = df["__label"].to_numpy().astype(int)
aux_cols = CONFIG["annotation"].get("aux_columns", []) or []
print(f"TEST samples: {len(y)}  |  Aux columns: {aux_cols}")

# Build inputs: image (N,H,W,C), flat (N,H*W*C), and optional aux (N,K) 
X_img, X_flat, X_aux = build_inputs(
    files, df,
    size=int(CONFIG["img_size"]),
    num_channels=int(CONFIG["num_channels"]),
    aux_cols=aux_cols
)
print("Inputs:", "X_img", X_img.shape, "| X_flat", X_flat.shape, "| X_aux", None if X_aux is None else X_aux.shape)

# Load Keras models (paths defined in CONFIG["models"]) 
models = {}
for name, path in CONFIG["models"].items():
    if Path(path).exists():
        try:
            models[name] = keras.models.load_model(path)
            print(f"✓ Loaded {name} from {path}")
            print("  Input specs:", describe_model_inputs(models[name]))
        except Exception as e:
            print(f"⚠️ Could not load {name}: {e}")
    else:
        print(f"⚠️ Missing model file: {path}")

# Evaluate: predict per model, threshold @ 0.5, collect metrics for Excel 
agg = BinaryEvalExcel(class_names=("clean","attack"))
for name, model in models.items():
    print(f"\n=== Evaluating {name} ===")
    # Assemble correct input(s) depending on model signatures (4D/3D/2D, single/multi-input)
    model_input = assemble_inputs(model, X_img, X_flat, X_aux)

    # Predict scores; accept shapes (N,), (N,1), (N,2) -> keep prob for positive class
    scores = model.predict(model_input, verbose=0).squeeze()
    if scores.ndim > 1:
        if scores.shape[-1] == 1:
            scores = scores[..., 0]
        else:
            scores = scores[..., -1]

    # Binary predictions at threshold 0.5
    y_pred = (scores >= 0.5).astype(int)

    # Register metrics (accuracy, precision, recall, f1, ROC-AUC, PR-AUC) + confusion counts
    agg.add(model=name, y_true=y, y_scores=scores, y_pred=y_pred)

# Write the Excel summary 
#excel_path = Path(out_dir) / "evaluation_detector_summary.xlsx"
#excel_path = Path(out_dir) / "evaluation_detector_unknown_fgsm_RNN_summary.xlsx" 
excel_path = Path(out_dir) / "evaluation_detector_unknown_PGD_summary.xlsx"
agg.save_excel(excel_path)

TEST samples: 46396  |  Aux columns: ['isNight']
Inputs: X_img (46396, 50, 50, 3) | X_flat (46396, 7500) | X_aux (46396, 1)
✓ Loaded ANN from models/ann_detector.keras
  Input specs: [{'shape': (None, 7501), 'rank': 2}]
✓ Loaded CNN from models/cnn_detector.keras
  Input specs: [{'shape': (None, 50, 50, 3), 'rank': 4}, {'shape': (None, 1), 'rank': 2}]
✓ Loaded RNN from models/rnn_detector.keras
  Input specs: [{'shape': (None, 50, 150), 'rank': 3}, {'shape': (None, 1), 'rank': 2}]
✓ Loaded SEMI_CNN from models/model_v_semi.keras
  Input specs: [{'shape': (None, 50, 50, 3), 'rank': 4}, {'shape': (None, 1), 'rank': 2}]
✓ Loaded SEMI_CNN_LIFELONG from models/teacher_ep3.keras
  Input specs: [{'shape': (None, 50, 50, 3), 'rank': 4}, {'shape': (None, 1), 'rank': 2}]

=== Evaluating ANN ===

=== Evaluating CNN ===

=== Evaluating RNN ===

=== Evaluating SEMI_CNN ===

=== Evaluating SEMI_CNN_LIFELONG ===


,model,n_samples,accuracy,precision,recall,f1,roc_auc,pr_auc
0,ANN,46396,0.743189,0.998041,0.658879,0.793748,0.926882,0.977609
1,CNN,46396,0.959910,0.999970,0.946576,0.972540,0.996437,0.998900
2,RNN,46396,0.876282,0.998148,0.836595,0.910259,0.981691,0.994406
3,SEMI_CNN,46396,0.991917,0.999449,0.989769,0.994585,0.999589,0.999874
4,SEMI_CNN_LIFELONG,46396,0.998448,0.998278,0.999655,0.998966,0.999924,0.999977


✓ Saved: evaluation_detector_unknown_PGD_summary.xlsx
